# 🌌 ICARUS_WORKER_ASTRONOMY — Astronomy & Astrophysics Worker
## Dedicated Remote Scientific Worker Runtime (Kaggle Account 3/4)

**Role:** Extragalactic Astronomy, Time-Domain Astrophysics & Stellar Photometry  
**Primary Model:** Zoobot 2.0 (`mwalmsley/zoobot-encoder-convnext_nano` / Galaxy Zoo)  
**Architecture:** ConvNeXt-Nano / EfficientNet encoder fine-tuned on DESI LS & SDSS Galaxy Zoo  
**Data Sources:** Sloan Digital Sky Survey (SDSS DR18), DESI Legacy Surveys, Kepler/TESS Light Curves  
**Deployment Profile:** Kaggle GPU (T4 or P100)

---

### Worker Responsibilities
- Hosts deterministic astronomy algorithms: **Light curve statistical moments, Lomb-Scargle periodogram periodicities, circular aperture photometry, CAS galaxy morphology asymmetry index**.
- Hosts **Zoobot 2.0** for galaxy morphology classification (smooth vs featured/disk, edge-on vs spiral arm count).
- Serves `/health`, `/capabilities`, `/analyze`, and `/unload-model` endpoints to `ICARUS_CORE`.


In [ ]:
# 1. Environment & Dependencies Setup
import os
import sys
from pathlib import Path

!pip install -q huggingface_hub astropy pyngrok uvicorn fastapi pydantic scipy timm

!git clone https://github.com/kanishk10-ux/ICARUS.git /kaggle/working/ICARUS 2>/dev/null || (cd /kaggle/working/ICARUS && git pull)
sys.path.insert(0, "/kaggle/working/ICARUS")

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Zoobot 2.0 Adapter Verification & Smoke Test
import numpy as np
from icarus.workers.astronomy.zoobot_adapter import ZoobotGalaxyAdapter
from icarus.algorithms.astronomy import AstronomyAlgorithms

device = "cuda" if torch.cuda.is_available() else "cpu"
adapter = ZoobotGalaxyAdapter(device=device)

ok, msg = adapter.load_model()
print("Load Status:", ok, "->", msg)

# Synthetic Galaxy Image: 3 channels (g, r, z), 224x224 pixels with exponential disk profile
x = np.linspace(-112, 112, 224)
y = np.linspace(-112, 112, 224)
xx, yy = np.meshgrid(x, y)
r = np.sqrt(xx**2 + (yy * 1.5)**2)
galaxy_flux = 1000.0 * np.exp(-r / 25.0) + np.random.normal(0, 5.0, (224, 224))
galaxy_rgb = np.stack([galaxy_flux * 0.8, galaxy_flux * 1.0, galaxy_flux * 1.2], axis=-1).astype(np.float32)

res = adapter.classify_morphology(galaxy_rgb, target_id="SDSS_J1234_SYNTH")
print("Zoobot Output Status:", res["status"])
if res["status"] == "COMPLETED":
    morph = res["morphology_probabilities"]
    print(f"Smooth: {morph['smooth_fraction']:.3f}, Featured/Disk: {morph['featured_or_disk_fraction']:.3f}, Edge-on: {morph['edge_on_fraction']:.3f}")

# Algorithmic Astronomy: Lomb-Scargle Periodogram
t_days = np.linspace(0, 30, 600)
flux = 1.0 + 0.05 * np.sin(2 * np.pi * t_days / 3.45) + np.random.normal(0, 0.005, 600)
period_res = AstronomyAlgorithms.compute_lomb_scargle_period(t_days, flux, min_period_days=0.5, max_period_days=10.0)
print(f"Detected Period: {period_res['dominant_period_days']:.4f} days (True: 3.4500 days, Peak Power: {period_res['peak_power']:.3f})")

In [ ]:
# 3. Launch Dedicated Worker Service with Public ngrok Tunnel
from scripts.worker_bootstrap import create_worker_app
import threading
import uvicorn

app = create_worker_app("astronomy", device=device)

def start_server():
    uvicorn.run(app, host="0.0.0.0", port=8004, log_level="warning")

thread = threading.Thread(target=start_server, daemon=True)
thread.start()
print("✅ Astronomy Worker running locally on port 8004")

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("NGROK_AUTH_TOKEN")
    from pyngrok import ngrok
    ngrok.set_auth_token(token)
    tunnel = ngrok.connect(8004, "http")
    public_url = tunnel.public_url
    print(f"🚀 PUBLIC NGROK ENDPOINT: {public_url}")
    print("\nUse this URL to register the worker with ICARUS_CORE:")
    print(f"Worker Registration Command for ICARUS_CORE:")
    print(f'curl -X POST <ICARUS_CORE_URL>/workers/register -H "Content-Type: application/json" -d \'{"worker_id": "worker-astronomy-01", "domain": "Astronomy", "endpoint": "{public_url}", "status": "READY", "capabilities": ["light_curve_moments", "lomb_scargle_periodogram", "aperture_photometry", "cas_asymmetry_index", "zoobot_morphology"]}\''')
except Exception as e:
    print("ngrok tunnel configuration note (Kaggle Secret NGROK_AUTH_TOKEN required for public access):", e)

In [ ]:
# 4. Worker Self-Test HTTP Loopback Verification
import requests

base_url = "http://127.0.0.1:8004"
health = requests.get(f"{base_url}/health").json()
print("Worker Health Status:", health)

caps = requests.get(f"{base_url}/capabilities").json()
print(f"Registered Capabilities: {len(caps['capabilities'])} operations supported")

# Test Lomb-Scargle via API
analysis_req = {
    "task_id": "test-periodogram-01",
    "operation": "lomb_scargle_periodogram",
    "parameters": {
        "time_days": t_days.tolist()[:100],
        "flux": flux.tolist()[:100],
        "min_period_days": 0.5,
        "max_period_days": 10.0
    }
}
analysis_resp = requests.post(f"{base_url}/analyze", json=analysis_req).json()
print("Lomb-Scargle Analysis Response:", analysis_resp)